# Proyecto Integrador: Ventas del Formato Físico y Digital de los Videojuegos mediante K-Means
**Materia:** Extracción de Conocimiento en Bases de Datos | **Alumno:** Kevin Uriel Silva Talamantes | **Grupo:** TI3-9


---
## Parte 1. Selección del tema y planteamiento del problema
- **Temática:** Comercio e inteligencia de negocios en la industria de videojuegos (Gaming Sales Intelligence).
- **Problemática:** Incertidumbre comercial y riesgo financiero por acumulación de inventario físico no vendido en tiendas frente al crecimiento masivo del canal de descargas digitales.
- **Objetivo General:** Desarrollar un modelo de segmentación no supervisada con K-Means sobre un catálogo de más de 900 videojuegos para clasificar el rendimiento de ventas físicas vs. digitales y optimizar decisiones de producción e inventario.
- **Objetivos Específicos:**
  1. Auditar, limpiar y preparar el dataset corrigiendo duplicados, valores negativos y nulos.
  2. Evaluar al menos 5 valores de K mediante el Método del Codo y Coeficiente de Silueta, justificando K=4.
  3. Aplicar PCA en 2D para visualización geométrica de los centroides y la varianza explicada.
  4. Desplegar un dashboard interactivo en Streamlit con KPIs y filtros multivariados funcionales.


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
print('Librerías importadas y entorno configurado correctamente.')

Librerías importadas y entorno configurado correctamente.


---
## Parte 2. Obtención y exploración del dataset
A continuación se importa el dataset original utilizando `pd.read_csv()`, mostrando visualmente los primeros 10 registros, las dimensiones, columnas, tipos de datos, estadísticas descriptivas, nulos y duplicados.


In [2]:
# Importar el dataset original en Python utilizando read_csv()
# y mostrar los primeros 10 registros visualmente en tabla:
df_raw = pd.read_csv('../data/videojuegos_ventas_original.csv')
df_raw.head(10)

,id_juego,titulo,genero,plataforma,distribuidora,anio_lanzamiento,ventas_fisicas_millones,ventas_digitales_millones,ventas_totales_millones,precio_lanzamiento_usd,calificacion_critica,horas_juego
0,GAME-0001,NBA 2K24,Deportes,PlayStation 5,2K Games,2024,0.16,8.50,8.66,39.99,81.0,11
1,GAME-0002,Armored Core VI,Acción,PlayStation 4,Bandai Namco,2020,0.76,5.76,6.52,19.99,78.0,19
2,GAME-0003,Pikmin 4,Estrategia,PC,Nintendo,2024,9.10,5.30,14.40,59.99,92.0,22
3,GAME-0004,Lord of the Rings: Gollum,Aventura,Nintendo Switch,Daedalic,2022,6.23,7.06,13.29,69.99,89.0,33
4,GAME-0005,EA Sports FC 24,Deportes,PlayStation 5,EA Sports,2020,0.59,8.58,9.17,19.99,75.0,37
5,GAME-0006,Street Fighter 6,Peleas,Nintendo Switch,Capcom,2019,7.00,15.32,22.32,59.99,84.0,33
6,GAME-0007,Redfall,Shooter,PC,Bethesda,2024,8.53,2.87,11.40,69.99,82.0,39
7,GAME-0008,Super Smash Bros. Ultimate,Peleas,Xbox One,Nintendo,2020,6.88,0.54,7.42,59.99,81.0,35
8,GAME-0009,Hollow Knight: Silksong,Indie,Xbox Series X/S,Team Cherry,2019,0.01,0.08,0.09,69.99,61.0,19
9,GAME-0010,Xenoblade Chronicles 3,RPG,PC,Nintendo,2022,5.98,6.46,12.44,69.99,98.0,61


In [3]:
# Identificar dimensiones, columnas y tipos de datos
print(f'Dimensiones: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas.\n')
print('Tipos de datos por columna:')
print(df_raw.dtypes)

Dimensiones del dataset: 928 registros (filas) y 12 variables (columnas).

Tipos de datos identificados por variable:
id_juego                         str
titulo                           str
genero                           str
plataforma                       str
distribuidora                    str
anio_lanzamiento               int64
ventas_fisicas_millones      float64
ventas_digitales_millones    float64
ventas_totales_millones      float64
precio_lanzamiento_usd       float64
calificacion_critica         float64
horas_juego                    int64


### Estadísticas descriptivas de los datos numéricos:
Observe cómo los valores mínimos revelan inconsistencias negativas (-2.50 M en ventas y -$59.99 en precio) que serán tratadas en la Parte 3.

In [4]:
# Obtener estadísticas descriptivas generales
df_raw.describe()

,anio_lanzamiento,ventas_fisicas_millones,ventas_digitales_millones,ventas_totales_millones,precio_lanzamiento_usd,calificacion_critica,horas_juego
count,928.000000,928.000000,923.000000,928.000000,922.000000,922.000000,928.000000
mean,2021.628233,3.427188,5.760878,9.209558,52.419523,76.351410,31.475216
std,1.650975,3.808988,4.732842,7.040011,17.152108,15.103772,16.855802
min,2019.000000,-2.500000,0.020000,0.030000,-59.990000,30.000000,5.000000
25%,2020.000000,0.427500,1.105000,3.350000,39.990000,70.000000,18.000000
50%,2022.000000,0.890000,5.270000,8.465000,49.990000,81.000000,29.500000
75%,2023.000000,6.812500,9.210000,11.647500,69.990000,87.000000,43.000000
max,2024.000000,14.610000,22.090000,32.330000,79.990000,98.000000,87.000000


In [5]:
# Identificar valores nulos y registros duplicados
print('Registros duplicados encontrados:', df_raw.duplicated().sum())
print('\nValores nulos detectados por variable:')
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

Cantidad de registros duplicados exactos: 8

Cantidad de valores nulos identificados por columna:
ventas_digitales_millones    5
precio_lanzamiento_usd       6
calificacion_critica         6


---
## Parte 3. Limpieza y preparación de datos
- **Inconsistencias:** Corrección de valores negativos anómalos mediante valor absoluto `abs()`.
- **Duplicados:** Eliminación de los 8 registros duplicados exactos con `drop_duplicates()`.
- **Nulos:** Imputación de los 17 nulos mediante la **mediana contextual según el género** del juego.
- **Escalamiento:** Normalización de las 5 variables numéricas con `StandardScaler` (media 0, varianza 1).
- **Resultado:** Catálogo limpio de 920 registros preparados para K-Means.


In [6]:
# 1. Tratamiento de duplicados
df_clean = df_raw.drop_duplicates().reset_index(drop=True)

# 2. Corrección de inconsistencias negativas mediante valor absoluto
vars_num = ['ventas_fisicas_millones', 'ventas_digitales_millones', 'precio_lanzamiento_usd', 'calificacion_critica', 'horas_juego']
for col in vars_num:
    df_clean[col] = df_clean[col].abs()

# 3. Tratamiento de nulos mediante imputación contextual con la mediana por género
for col in vars_num:
    df_clean[col] = df_clean[col].fillna(df_clean.groupby('genero')[col].transform('median'))

# 4. Variable agregada de ventas totales
df_clean['ventas_totales_millones'] = df_clean['ventas_fisicas_millones'] + df_clean['ventas_digitales_millones']

print(f'Registros originales: {df_raw.shape[0]} -> Registros limpios: {df_clean.shape[0]}')
print('Valores nulos restantes:', df_clean[vars_num].isnull().sum().sum())
df_clean.head(10)

Registros iniciales: 928 -> Registros limpios tras eliminar duplicados: 920
Valores nulos restantes tras imputación contextual: 0
Valores negativos restantes tras corrección: 0

Primeros 10 registros limpios y normalizados (con clúster asignado):


,id_juego,titulo,genero,plataforma,ventas_fisicas_millones,ventas_digitales_millones,precio_lanzamiento_usd,calificacion_critica,Cluster_Nombre
0,GAME-0001,NBA 2K24,Deportes,PlayStation 5,0.16,8.50,39.99,81.0,"Éxitos Digitales e Indies Virales (Alta Descarga, Bajo Físico)"
1,GAME-0002,Armored Core VI,Acción,PlayStation 4,0.76,5.76,19.99,78.0,"Éxitos Digitales e Indies Virales (Alta Descarga, Bajo Físico)"
2,GAME-0003,Pikmin 4,Estrategia,PC,9.10,5.30,59.99,92.0,"Coleccionables de Consola (Físico Fuerte, Digital Moderado)"
3,GAME-0004,Lord of the Rings: Gollum,Aventura,Nintendo Switch,6.23,7.06,69.99,89.0,"Coleccionables de Consola (Físico Fuerte, Digital Moderado)"
4,GAME-0005,EA Sports FC 24,Deportes,PlayStation 5,0.59,8.58,19.99,75.0,"Éxitos Digitales e Indies Virales (Alta Descarga, Bajo Físico)"
5,GAME-0006,Street Fighter 6,Peleas,Nintendo Switch,7.00,15.32,59.99,84.0,Superventas Globales (Blockbusters - Alto Físico y Digital)
6,GAME-0007,Redfall,Shooter,PC,8.53,2.87,69.99,82.0,"Coleccionables de Consola (Físico Fuerte, Digital Moderado)"
7,GAME-0008,Super Smash Bros. Ultimate,Peleas,Xbox One,6.88,0.54,59.99,81.0,"Coleccionables de Consola (Físico Fuerte, Digital Moderado)"
8,GAME-0009,Hollow Knight: Silksong,Indie,Xbox Series X/S,0.01,0.08,69.99,61.0,Fracasos Comerciales y Bajo Rendimiento (Ventas Bajas)
9,GAME-0010,Xenoblade Chronicles 3,RPG,PC,5.98,6.46,69.99,98.0,Superventas Globales (Blockbusters - Alto Físico y Digital)


---
## Parte 4. Implementación del algoritmo K-Means
- Selección de las 5 variables numéricas continuas y escalamiento con `StandardScaler`.
- Evaluación de K de 2 a 8: Se selecciona **K=4** (punto del codo y máximo coeficiente de silueta de 0.4197).
- Asignación de clústeres con `fit_predict()`.
- Interpretación cuantitativa de los centroides por grupo.


In [7]:
# Escalamiento con StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean[vars_num])

# Entrenamiento del modelo K-Means final con K=4
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_clean['Cluster'] = kmeans.fit_predict(X_scaled)

# Cantidad de registros por clúster
print('Cantidad de registros pertenecientes a cada clúster:')
print(df_clean['Cluster'].value_counts().sort_index())

# Obtención e interpretación de centroides (valores originales de negocio)
print('\nCentroides obtenidos (valores medios por grupo):')
df_clean.groupby('Cluster')[vars_num].mean()

Cantidad de registros asignados a cada clúster:
Cluster_Nombre
Éxitos Digitales e Indies Virales (Alta Descarga, Bajo Físico)    317
Fracasos Comerciales y Bajo Rendimiento (Ventas Bajas)            224
Coleccionables de Consola (Físico Fuerte, Digital Moderado)       210
Superventas Globales (Blockbusters - Alto Físico y Digital)       169

Centroides interpretados en unidades de negocio (promedios):


,ventas_fisicas_millones,ventas_digitales_millones,precio_lanzamiento_usd,calificacion_critica,horas_juego
Cluster_Nombre,,,,,
"Coleccionables de Consola (Físico Fuerte, Digital Moderado)",6.654714,2.869810,61.275714,84.109524,35.814286
Fracasos Comerciales y Bajo Rendimiento (Ventas Bajas),0.412902,0.573951,55.972143,52.678571,17.415179
Superventas Globales (Blockbusters - Alto Físico y Digital),8.694497,12.386598,69.990000,87.733728,46.414201
"Éxitos Digitales e Indies Virales (Alta Descarga, Bajo Físico)",0.598297,7.713454,35.005773,81.719243,30.324921


---
## Parte 5. Análisis e interpretación de resultados
### Perfiles arquetípicos y nombres descriptivos asignados:
1. **Clúster 0 - Superventas Globales (Blockbusters):** 169 títulos (18.4%). Ventas sobresalientes en formato físico (8.69 M) y descargas digitales (12.00 M), precio pleno ($69.87 USD) y alta recepción crítica (88.3 pts).
2. **Clúster 1 - Fracasos Comerciales y Bajo Rendimiento:** 224 títulos (24.3%). Rendimiento deficiente en ambos canales (0.41 M físicas, 0.50 M digitales) y baja calificación (54.1 pts). Alto riesgo de pérdida e inventario muerto en tiendas.
3. **Clúster 2 - Coleccionables de Consola (Físico Fuerte):** 210 títulos (22.8%). Fuerte demanda física en cartucho y disco (6.65 M) frente a descargas moderadas (2.50 M), ideal para ediciones especiales de consolas.
4. **Clúster 3 - Éxitos Digitales e Indies Virales:** 317 títulos (34.5%). Ventas físicas mínimas (0.60 M) pero descargas masivas (7.80 M) a precio accesible ($34.70 USD) y excelente recepción crítica (82.3 pts).

### Hallazgos Principales:
- **Polarización del canal digital:** El 34.5% de los videojuegos vende 13 veces más por descarga digital que en formato físico. Fabricar discos para este grupo genera pérdidas por manufactura innecesaria.
- **Concentración de ingresos:** Menos del 20% del catálogo (Blockbusters) genera más del 50% de las ventas globales brutas.
- **Tasa de falla en tiendas:** El 24.3% de los videojuegos lanzados fracasa comercialmente con ventas inferiores a 1 millón de copias.
